### **Clone repository và thiết lập ban đầu**
- Clone source code từ GitHub về môi trường Colab
- Di chuyển vào thư mục project
- Copy file `.env.example` thành `.env` để cấu hình biến môi trường

In [1]:
!git clone https://github.com/dauvannam1804/data-agent-demo.git
%cd data-agent-demo
!cp .env.example .env

Cloning into 'data-agent-demo'...
remote: Enumerating objects: 366, done.
remote: Counting objects: 100% (366/366), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 366 (delta 130), reused 303 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (366/366), 15.92 MiB | 6.85 MiB/s, done.
Resolving deltas: 100% (130/130), done.
Updating files: 100% (139/139), done.
/content/data-agent-demo


### **Tạo môi trường ảo và cài đặt thư viện**
- Tạo virtual environment bằng `uv`
- Kích hoạt môi trường ảo
- Cài đặt toàn bộ dependencies từ file `requirements.txt`

In [2]:
!uv venv
!source .venv/bin/activate
!uv add -r requirements.txt

Using CPython 3.13.12
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 116 packages in 15ms
Prepared 114 packages in 8.11s
Installed 114 packages in 359ms
 + agno==2.5.10
 + altair==6.0.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + attrs==25.4.0
 + bcrypt==5.0.0
 + blinker==1.9.0
 + build==1.4.0
 + cachetools==7.0.5
 + certifi==2026.2.25
 + charset-normalizer==3.4.6
 + chromadb==1.5.5
 + click==8.3.1
 + contourpy==1.3.3
 + cycler==0.12.1
 + distro==1.9.0
 + docstring-parser==0.17.0
 + duckdb==1.5.0
 + durationpy==0.10
 + filelock==3.25.2
 + flatbuffers==25.12.19
 + fonttools==4.62.1
 + fsspec==2026.2.0
 + gitdb==4.0.12
 + gitpython==3.1.46
 + googleapis-common-protos==1.73.0
 + grpcio==1.78.0
 + h11==0.16.0
 + h2==4.3.0
 + hf-xet==1.4.2
 + hpack==4.1.0
 + httpcore==1.0.9
 + httptools==0.7.1
 + httpx==0.28.1
 + huggingface-hub==1.7.1
 + hyperframe==6.1.0
 + idna==3.11
 + importlib-metadata==8.7.1
 + importlib-resources==6

### Cài đặt pyngrok để public ứng dụng
- Cài đặt thư viện `pyngrok` để expose local server ra internet
- Sử dụng để truy cập Streamlit app từ bên ngoài Colab

In [3]:
!source .venv/bin/activate
!uv pip install pyngrok

Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 43ms
Prepared 1 package in 10ms
Installed 1 package in 6ms
 + pyngrok==7.5.1


### **Cấu hình Ngrok Auth Token**
- Mở Colab secrets (nhần vào biểu tượng dạng 🔑 ở bên trái)
- `Add new secret` với Name `NGROKName_AUTH_TOKEN` và Value là `ngrok auth token` tại [đây](https://dashboard.ngrok.com/get-started/your-authtoken)
- Token này dùng để xác thực và sử dụng dịch vụ ngrok

In [4]:
!source .venv/bin/activate
import os
from google.colab import userdata
from pyngrok import ngrok
NGROK_AUTH_TOKEN = userdata.get('NGROKName_AUTH_TOKEN')

### **Chạy ứng dụng Streamlit (Baseline)**
- Chạy app `baseline_system/app_baseline.py` ở background
- Sử dụng `nohup` để giữ process chạy ngầm
- Log được ghi vào file `streamlit.log`

In [5]:
!source .venv/bin/activate
get_ipython().system_raw('cd /content/data-agent-demo && nohup ./.venv/bin/streamlit run baseline_system/app_baseline.py --server.port 8501 > streamlit.log 2>&1 &')
print("Đã khởi động Streamlit")

Đã khởi động Streamlit


### **Expose ứng dụng qua Ngrok**
- Kết nối ngrok tới port 8501
- Tạo URL public để truy cập Streamlit app từ internet

In [6]:
!source .venv/bin/activate
# 4. Kích hoạt đường hầm siêu tốc Ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
http_tunnel = ngrok.connect(8501)
print(f"🚀 Giao diện Streamlit của bạn đã sẵn sàng tại: {http_tunnel.public_url}")

🚀 Giao diện Streamlit của bạn đã sẵn sàng và 100% không dính lỗi tại: https://c16f-34-48-255-198.ngrok-free.app


### **Dừng ứng dụng và giải phóng tài nguyên**
- Kill toàn bộ process Streamlit đang chạy
- Tắt kết nối ngrok

In [18]:
!pkill -f streamlit
from pyngrok import ngrok
ngrok.kill()

### **Xây dựng schema metadata**
- Chạy script `schema_builder.py`
- Dùng để tạo schema cho hệ thống query (Query GPT System)

In [14]:
!/content/data-agent-demo/.venv/bin/python /content/data-agent-demo/query_gpt_system/metadata/schema_builder.py

INFO:__main__:Scanning directory: /content/data-agent-demo/data-code/InfiAgent/da-dev-tables for CSV files...
INFO:__main__:Successfully extracted schema for hotel_data.csv
INFO:__main__:Successfully extracted schema for cms_hospital_readmissions.csv
INFO:__main__:Successfully extracted schema for unemployement_industry.csv
INFO:__main__:Successfully extracted schema for auto-mpg.csv
INFO:__main__:Successfully extracted schema for beauty and the labor market.csv
INFO:__main__:Successfully extracted schema for fb_articles_20180822_20180829_df.csv
INFO:__main__:Successfully extracted schema for baseball_data.csv
INFO:__main__:Successfully extracted schema for traj-Osak.csv
INFO:__main__:Successfully extracted schema for ravenna_250715.csv
INFO:__main__:Successfully extracted schema for GODREJIND.csv
INFO:__main__:Successfully extracted schema for credit-data-post-import.csv
INFO:__main__:Successfully extracted schema for diamonds.csv
INFO:__main__:Successfully extracted schema for countr

### **Nạp dữ liệu mẫu SQL**
- Chạy script `ingest_samples.py`
- Import các câu query mẫu vào hệ thống để phục vụ truy vấn

In [15]:
!/content/data-agent-demo/.venv/bin/python /content/data-agent-demo/query_gpt_system/sql_samples/ingest_samples.py

👉 Tìm thấy 257 câu hỏi mẫu để lập chỉ mục (index).
🚀 Đang đưa 257 bản ghi vào ChromaDB tại query_gpt_system/sql_samples/chroma_db...
INFO Upserting 257 documents                                                    
✅ Hoàn tất: Các câu SQL mẫu đã được lập chỉ mục thành công.


### **Chạy ứng dụng Query GPT**
- Chạy app `query_gpt_system/app_query_gpt.py`
- Đây là giao diện chính để query dữ liệu bằng GPT

In [19]:
!source .venv/bin/activate
get_ipython().system_raw('cd /content/data-agent-demo && nohup ./.venv/bin/streamlit run query_gpt_system/app_query_gpt.py --server.port 8501 > streamlit.log 2>&1 &')
print("Đã khởi động Streamlit")

Đã khởi động Streamlit


### **Expose Query GPT app qua Ngrok**

In [20]:
!source .venv/bin/activate
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
http_tunnel = ngrok.connect(8501)
print(f"🚀 Giao diện Streamlit của bạn đã sẵn sàng tại: {http_tunnel.public_url}")

🚀 Giao diện Streamlit của bạn đã sẵn sàng và 100% không dính lỗi tại: https://8bd7-34-48-255-198.ngrok-free.app
